# 02 The cutting workflow

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsc-wdc/qdislib/blob/main/notebooks/02_cutting_workflow.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/bsc-wdc/qdislib/main?labpath=notebooks/02_cutting_workflow.ipynb)

`gate_cutting` does everything in one call. The **three-step workflow** splits that call in
three, so the middle one is yours:

1. **generate** the subcircuits;
2. **run** each one, wherever you choose;
3. **reconstruct** the result of the original circuit.

Use it when you want to send different subcircuits to different places, inspect them, or run them with your own tools.

In [1]:
# Installs Qdislib on Colab or Binder; does nothing where Qdislib 2 is installed.
try:
    import Qdislib
    assert Qdislib.__version__.startswith("2.")
except (ImportError, AssertionError):
    %pip install -q "qdislib[qiskit]>=2.0"

In [2]:
from qiskit import QuantumCircuit


def example_circuit():
    """Five qubits in a line, joined by four CZ gates."""
    qc = QuantumCircuit(5)
    for qubit, angle in enumerate([0.8, 0.5, 0.6, 0.4, 0.3]):
        qc.ry(angle, qubit)
    for qubit in range(4):
        qc.cz(qubit, qubit + 1)
    for qubit, angle in enumerate([0.2, 0.1, 0.7, 0.3, 0.9]):
        qc.rx(angle, qubit)
    return qc


circuit = example_circuit()
print(circuit.draw(fold=90))


import Qdislib as qd

cut = ["CZ_7"]
exact = qd.analytical_solution(circuit, "ZZZZZ")

     ┌─────────┐   ┌─────────┐                                 
q_0: ┤ Ry(0.8) ├─■─┤ Rx(0.2) ├─────────────────────────────────
     ├─────────┤ │ └─────────┘┌─────────┐                      
q_1: ┤ Ry(0.5) ├─■──────■─────┤ Rx(0.1) ├──────────────────────
     ├─────────┤        │     └─────────┘┌─────────┐           
q_2: ┤ Ry(0.6) ├────────■──────────■─────┤ Rx(0.7) ├───────────
     ├─────────┤                   │     └─────────┘┌─────────┐
q_3: ┤ Ry(0.4) ├───────────────────■──────────■─────┤ Rx(0.3) ├
     ├─────────┤                              │     ├─────────┤
q_4: ┤ Ry(0.3) ├──────────────────────────────■─────┤ Rx(0.9) ├
     └─────────┘                                    └─────────┘


## 1. Generate

Every cut type has a `_subcircuits` form returning the same kind of list: `gate_cutting_subcircuits`,
`wire_cutting_subcircuits`, `gate_cutting_hadamard_subcircuits` and `circuit_cutting_subcircuits`.
The list and each item print what they hold.

In [3]:
subcircuits = qd.gate_cutting_subcircuits(circuit, cut, "ZZZZZ")
print(subcircuits)
print(subcircuits[0])
print("qubits:", subcircuits[0].num_qubits)
print(subcircuits[0].circuit.draw(fold=70))

<CutCircuits: 12 subcircuits, 2 fragments, gate ['CZ_7'], gamma 3, observable ZZZZZ>
<Subcircuit: fragment 0, 2 qubits [q0 q1], CZ_7 side 1: RZ(-π/2)>
qubits: 2
     ┌─────────┐   ┌─────────┐            
q_0: ┤ Ry(0.8) ├─■─┤ Rx(0.2) ├────────────
     ├─────────┤ │ ├─────────┴┐┌─────────┐
q_1: ┤ Ry(0.5) ├─■─┤ Rz(-π/2) ├┤ Rx(0.1) ├
     └─────────┘   └──────────┘└─────────┘
c: 2/═════════════════════════════════════
                                          


The list holds no SDK choice: `s.circuit` is built when asked for, in the input circuit's
SDK by default. A gate cut's measuring side has a mid-circuit measurement, which is what the `m`
in the circuit below is.

In [4]:
print(subcircuits[4])
print(subcircuits[4].circuit.draw(fold=70))

<Subcircuit: fragment 0, 2 qubits [q0 q1], CZ_7 side 1: measure, RZ(-π/2)>
     ┌─────────┐   ┌─────────┐                       
q_0: ┤ Ry(0.8) ├─■─┤ Rx(0.2) ├───────────────────────
     ├─────────┤ │ └───┬─┬───┘┌──────────┐┌─────────┐
q_1: ┤ Ry(0.5) ├─■─────┤M├────┤ Rz(-π/2) ├┤ Rx(0.1) ├
     └─────────┘       └╥┘    └──────────┘└─────────┘
c: 2/═══════════════════╩════════════════════════════
                        1                            


## 2. Run

`qd.expectation_value` runs one subcircuit and returns a `FragmentResult`. It takes `shots`, `backend`
and `seed`. This list comprehension is where a distributed or device-specific loop goes: choose `backend=`
per subcircuit with `s.num_qubits`, batch, or reuse earlier results.

In [5]:
values = [qd.expectation_value(s, shots=8192, seed=i) for i, s in enumerate(subcircuits)]
print(values[0])

FragmentResult(Z[q0,q1]: +0.545654; 1 correlator)


## 3. Reconstruct

`reconstruct` takes one result per subcircuit, in order. `describe` shows what happened: which
fragment and cut term each subcircuit ran, and what it measured.

In [6]:
print(f"reconstructed {subcircuits.reconstruct(values):+.4f}   exact {exact:+.4f}")
subcircuits.describe(values)

reconstructed +0.2465   exact +0.2390


#,fragment,qubits,terms,values
0,0,q0 q1,CZ_7 side 1: RZ(-π/2),"Z[q0,q1]: +0.5457"
1,1,q2 q3 q4,CZ_7 side 2: RZ(-π/2),"Z[q2,q3,q4]: +0.1362"
2,0,q0 q1,"CZ_7 side 1: Z, RZ(-π/2)","Z[q0,q1]: +0.6538"
3,1,q2 q3 q4,"CZ_7 side 2: Z, RZ(-π/2)","Z[q2,q3,q4]: +0.5659"
4,0,q0 q1,"CZ_7 side 1: measure, RZ(-π/2)","Z[q0,q1]: -0.6812"
5,1,q2 q3 q4,CZ_7 side 2: RZ(-π),"Z[q2,q3,q4]: +0.3115"
6,0,q0 q1,"CZ_7 side 1: measure, RZ(-π/2)","Z[q0,q1]: -0.6709"
7,1,q2 q3 q4,CZ_7 side 2: identity,"Z[q2,q3,q4]: +0.3826"
8,0,q0 q1,CZ_7 side 1: RZ(-π),"Z[q0,q1]: +0.5938"
9,1,q2 q3 q4,"CZ_7 side 2: measure, RZ(-π/2)","Z[q2,q3,q4]: -0.4224"


## The same three steps for any cut

Only the generator changes. A wire cut's prepared line is shown as `q2'`, and a Hadamard
cut's ancilla as `a0`.

In [7]:
wire = qd.wire_cutting_subcircuits(circuit, [("CZ_7", "CZ_8")], "ZZZZZ")
had = qd.gate_cutting_hadamard_subcircuits(circuit, cut, "ZZZZZ")
mix = qd.circuit_cutting_subcircuits(circuit, {"gate": ["CZ_6"], "wire": [("CZ_8", "CZ_9")]}, "ZZZZZ")
for name, subs in [("wire", wire), ("Hadamard", had), ("mixed", mix)]:
    result = subs.reconstruct([qd.expectation_value(s, shots=8192, seed=1) for s in subs])
    print(f"{name:9s} {subs}\n          -> {result:+.4f}")
print(wire[2])
print(had[0])

wire      <CutCircuits: 16 subcircuits, 2 fragments, wire [('CZ_7', 'CZ_8')], gamma 4, observable ZZZZZ>
          -> +0.2393
Hadamard  <CutCircuits: 12 subcircuits, 2 fragments, Hadamard ['CZ_7'], gamma 3, observable ZZZZZ>
          -> +0.2305


mixed     <CutCircuits: 144 subcircuits, 3 fragments, gate ['CZ_6']; wire [('CZ_8', 'CZ_9')], gamma 12, observable ZZZZZ>
          -> +0.2356
<Subcircuit: fragment 0, 3 qubits [q0 q1 q2], wire CZ_7→CZ_8: measure Z>
<Subcircuit: fragment 0, 3 qubits [q0 q1 a0], Hadamard CZ_7: term (0,0), parity 0>


## Run the circuits with your own tools

`subs.circuits(software)` gives just the native circuits, in `"qiskit"`, `"qibo"`, `"cudaq"`, `"pennylane"`
or `"qilisdk"`. With `defer_measurements=True` a mid-circuit measurement moves onto an extra qubit
read at the end, for hardware or simulators without mid-circuit measurement.

Their counts come back with `reconstruct_counts`: one `{bitstring: count}` per circuit, every qubit measured
at the end. `bit_order="q0_last"` is the order of Qiskit's `get_counts()`.

In [8]:
from qiskit_aer import AerSimulator

simulator = AerSimulator(seed_simulator=1)
circuits = subcircuits.circuits("qiskit", defer_measurements=True)
print(len(circuits), "circuits; the first has", circuits[0].num_qubits, "qubits,",
      "the measuring side", circuits[4].num_qubits)

counts = []
for qc in circuits:
    measured = qc.copy()
    measured.measure_all()
    counts.append(simulator.run(measured, shots=8192).result().get_counts())

print(counts[0])
print(f"reconstructed {subcircuits.reconstruct_counts(counts, bit_order='q0_last'):+.4f}   exact {exact:+.4f}")

12 circuits; the first has 2 qubits, the measuring side 3
{'11 00': 58, '01 00': 1244, '10 00': 616, '00 00': 6274}
reconstructed +0.2351   exact +0.2390


`qd.result_from_counts(s, counts)` converts one subcircuit's counts, so that results of your own runs and
of Qdislib's can be mixed in one `reconstruct`.

In [9]:
mixed = [
    qd.result_from_counts(s, c, bit_order="q0_last") if i % 2 else qd.expectation_value(s, shots=8192, seed=i)
    for i, (s, c) in enumerate(zip(subcircuits, counts))
]
print(f"reconstructed {subcircuits.reconstruct(mixed):+.4f}")

reconstructed +0.2358


Under PyCOMPSs each `expectation_value` in step 2 is a task and `values` holds futures: the same code runs in parallel
(notebook 09).

Next: **03 Finding cuts**.